In [1]:
import re
import numpy as np
import pandas as pd
import nltk

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

import ipywidgets as widgets
from IPython.display import display, clear_output

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

True

In [2]:
training_data = [
    # Intent 1: blood_sugar_monitoring (8 queries)
    ("How often should I check my blood sugar levels?", "blood_sugar_monitoring"),
    ("What is a normal fasting blood glucose range?", "blood_sugar_monitoring"),
    ("Should I test my blood sugar before and after meals?", "blood_sugar_monitoring"),
    ("How do I use a continuous glucose monitor?", "blood_sugar_monitoring"),
    ("Why is my morning fasting glucose high?", "blood_sugar_monitoring"),
    ("What does postprandial blood sugar mean?", "blood_sugar_monitoring"),
    ("How should I log my daily glucose readings?", "blood_sugar_monitoring"),
    ("When is the best time to check blood glucose?", "blood_sugar_monitoring"),

    # Intent 2: diet_and_nutrition (8 queries)
    ("What foods help keep blood sugar steady?", "diet_and_nutrition"),
    ("How does carbohydrate counting work for diabetes?", "diet_and_nutrition"),
    ("What is the glycemic index and why does it matter?", "diet_and_nutrition"),
    ("Are whole grains better than refined grains for glucose control?", "diet_and_nutrition"),
    ("How much fiber should I include in my meals?", "diet_and_nutrition"),
    ("Can people with diabetes eat fruit?", "diet_and_nutrition"),
    ("How can I plan balanced meals to avoid sugar spikes?", "diet_and_nutrition"),
    ("What beverages are safest for managing blood glucose?", "diet_and_nutrition"),

    # Intent 3: medication_and_insulin (8 queries)
    ("How does metformin help manage blood sugar?", "medication_and_insulin"),
    ("What is the difference between basal and bolus insulin?", "medication_and_insulin"),
    ("What should I do if I miss a scheduled diabetes dose?", "medication_and_insulin"),
    ("How should insulin pens or vials be stored?", "medication_and_insulin"),
    ("Can oral diabetes medications cause stomach upset?", "medication_and_insulin"),
    ("Why do insulin doses sometimes need adjustment?", "medication_and_insulin"),
    ("How do GLP-1 medications support glucose management?", "medication_and_insulin"),
    ("What is A1C and how does treatment affect it?", "medication_and_insulin"),

    # Intent 4: physical_activity (8 queries)
    ("What exercises are safe for someone with diabetes?", "physical_activity"),
    ("How does walking after meals affect blood sugar?", "physical_activity"),
    ("Should I check glucose before starting a workout?", "physical_activity"),
    ("How can I prevent low blood sugar during exercise?", "physical_activity"),
    ("Is strength training helpful for insulin sensitivity?", "physical_activity"),
    ("How long should a beginner exercise session be?", "physical_activity"),
    ("What should I carry with me when exercising?", "physical_activity"),
    ("How does intense exercise change blood glucose?", "physical_activity"),

    # Intent 5: hypoglycemia_management (8 queries)
    ("What are the early signs of low blood sugar?", "hypoglycemia_management"),
    ("How do I treat mild hypoglycemia quickly?", "hypoglycemia_management"),
    ("What does the 15-15 rule mean for low glucose?", "hypoglycemia_management"),
    ("Why do shakiness and sweating happen during low blood sugar?", "hypoglycemia_management"),
    ("What causes nighttime hypoglycemia?", "hypoglycemia_management"),
    ("How do I recheck blood sugar after treating a low?", "hypoglycemia_management"),
    ("What snacks help prevent dropping blood sugar?", "hypoglycemia_management"),
    ("Why is skipping meals risky when taking insulin?", "hypoglycemia_management"),

    # Intent 6: emergency_escalation (8 queries)
    ("I have severe confusion and extreme dizziness with low blood sugar", "emergency_escalation"),
    ("My blood sugar is dangerously high and I am vomiting", "emergency_escalation"),
    ("I cannot stay awake and my breathing feels laboured", "emergency_escalation"),
    ("Fainting and chest pain after taking insulin", "emergency_escalation"),
    ("Severe abdominal pain, rapid breathing, and fruity breath", "emergency_escalation"),
    ("Unresponsive person with suspected severe hypoglycemia", "emergency_escalation"),
    ("Persistent vomiting and inability to keep fluids down with high ketones", "emergency_escalation"),
    ("Sudden loss of consciousness or seizure with low glucose", "emergency_escalation"),
]

df_corpus = pd.DataFrame(training_data, columns=["query", "intent"])
print(f"Corpus size: {len(df_corpus)} queries across {df_corpus['intent'].nunique()} intents.")

Corpus size: 48 queries across 6 intents.


In [3]:
taxonomy = {
    "Glycemic Monitoring & Metrics": {
        "Self-Monitoring of Blood Glucose": ["Fasting glucose", "Postprandial glucose", "Meter logging"],
        "Continuous & Long-Term Tracking": ["CGM sensors", "Glycemic trends", "HbA1c targets"]
    },
    "Lifestyle & Daily Self-Care": {
        "Nutrition & Glycemic Impact": ["Carbohydrate counting", "Fiber & whole foods", "Glycemic response"],
        "Physical Activity & Recovery": ["Aerobic exercise", "Resistance training", "Workout glucose safety"]
    },
    "Pharmacotherapy & Insulin Care": {
        "Oral & Non-Insulin Therapies": ["Metformin", "GLP-1 receptor agonists", "Adherence & side effects"],
        "Insulin Regimens & Handling": ["Basal vs. bolus insulin", "Cold/room storage", "Dose timing"]
    },
    "Acute Complications & Safety": {
        "Hypoglycemia Care": ["Early warning signs", "15-15 rule", "Fast-acting carbs"],
        "Severe & Emergency Escalation": ["Severe hypoglycemia", "DKA / hyperglycemic crisis", "Urgent medical help"]
    }
}

In [4]:
domain_concepts = [
    "Blood Glucose",
    "Insulin",
    "Carbohydrates",
    "Glycemic Index",
    "Physical Activity",
    "Hypoglycemia",
    "HbA1c",
    "Hyperglycemia / DKA"
]

concept_relationships = [
    ("Carbohydrates", "INCREASES", "Blood Glucose"),
    ("Insulin", "LOWERS", "Blood Glucose"),
    ("Physical Activity", "IMPROVES_SENSITIVITY_TO", "Insulin"),
    ("Physical Activity", "CAN_REDUCE", "Blood Glucose"),
    ("Glycemic Index", "INDICATES_RATE_OF_RISE_IN", "Blood Glucose"),
    ("Excess Insulin", "CAN_TRIGGER", "Hypoglycemia"),
    ("Hypoglycemia", "TREATED_WITH", "Fast-Acting Carbohydrates"),
    ("Sustained High Blood Glucose", "ELEVATES", "HbA1c"),
    ("Severe Insulin Deficiency", "PREDISPOSES_TO", "Hyperglycemia / DKA"),
    ("Blood Glucose Monitoring", "GUIDES_ADJUSTMENT_OF", "Insulin & Diet")
]

In [5]:
stop_words = set(stopwords.words("english"))
# Keep negation/safety cues out of stopword removal if helpful, while using NLTK stopwords
lemmatizer = WordNetLemmatizer()

def preprocess_text(text: str):
    # 1. Lowercase & keep alphabetic tokens/spaces
    text_lower = text.lower()
    text_clean = re.sub(r"[^a-z\s]", " ", text_lower)
    # 2. Tokenize
    tokens = word_tokenize(text_clean)
    # 3. Stopword removal
    no_stops = [tok for tok in tokens if tok not in stop_words and len(tok) > 1]
    # 4. Lemmatization
    lemmatized = [lemmatizer.lemmatize(tok) for tok in no_stops]
    return {
        "cleaned": text_clean.strip(),
        "tokens": tokens,
        "filtered_tokens": no_stops,
        "lemmas": lemmatized,
        "processed_str": " ".join(lemmatized)
    }

In [6]:
X_processed = [preprocess_text(q)["processed_str"] for q in df_corpus["query"]]
y_labels = df_corpus["intent"].values

X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y_labels, test_size=0.25, random_state=42, stratify=y_labels
)

vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

classifier = LogisticRegression(max_iter=1000, random_state=42)
classifier.fit(X_train_tfidf, y_train)

# Retrain on full corpus for deployment after evaluating holdout performance
y_pred_test = classifier.predict(X_test_tfidf)
acc = accuracy_score(y_test, y_pred_test)
prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred_test, average="weighted", zero_division=0)
cm = confusion_matrix(y_test, y_pred_test, labels=classifier.classes_)

print("\n--- Holdout Evaluation Metrics ---")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-Score : {f1:.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred_test, zero_division=0))
print("Confusion Matrix (rows=true, cols=pred):\n", pd.DataFrame(cm, index=classifier.classes_, columns=classifier.classes_))

# Fit final model on full dataset for richer runtime intent coverage
full_vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
X_full_tfidf = full_vectorizer.fit_transform(X_processed)
final_classifier = LogisticRegression(max_iter=1000, random_state=42)
final_classifier.fit(X_full_tfidf, y_labels)


--- Holdout Evaluation Metrics ---
Accuracy : 0.5833
Precision: 0.4444
Recall   : 0.5833
F1-Score : 0.4667

Classification Report:
                          precision    recall  f1-score   support

 blood_sugar_monitoring       0.50      1.00      0.67         2
     diet_and_nutrition       0.00      0.00      0.00         2
   emergency_escalation       1.00      0.50      0.67         2
hypoglycemia_management       0.50      1.00      0.67         2
 medication_and_insulin       0.67      1.00      0.80         2
      physical_activity       0.00      0.00      0.00         2

               accuracy                           0.58        12
              macro avg       0.44      0.58      0.47        12
           weighted avg       0.44      0.58      0.47        12

Confusion Matrix (rows=true, cols=pred):
                          blood_sugar_monitoring  diet_and_nutrition  \
blood_sugar_monitoring                        2                   0   
diet_and_nutrition            

LogisticRegression(max_iter=1000, random_state=42)

In [7]:
knowledge_base = {
    "blood_sugar_monitoring": {
        "title": "Blood Glucose Monitoring",
        "guidance": (
            "Regular glucose checks—such as fasting, pre-meal, and 1–2 hours post-meal—help show "
            "how food, activity, and medication affect your levels. Keeping a consistent log with "
            "timestamps and context makes patterns easier for you and your clinician to review."
        ),
        "related_concepts": ["Blood Glucose", "HbA1c"]
    },
    "diet_and_nutrition": {
        "title": "Diabetes Nutrition & Meal Balance",
        "guidance": (
            "Pairing non-starchy vegetables, lean protein, and healthy fats with measured portions of "
            "high-fiber carbohydrates or lower-glycemic foods helps moderate post-meal glucose rises. "
            "Water and unsweetened drinks are generally preferred over sugary beverages."
        ),
        "related_concepts": ["Carbohydrates", "Glycemic Index", "Blood Glucose"]
    },
    "medication_and_insulin": {
        "title": "Diabetes Medications & Insulin Basics",
        "guidance": (
            "Medications like metformin and insulin work through distinct pathways to support glucose "
            "regulation. Always follow your prescribed schedule and storage instructions, and check with "
            "your healthcare provider or pharmacist before changing doses or handling missed doses."
        ),
        "related_concepts": ["Insulin", "Blood Glucose", "HbA1c"]
    },
    "physical_activity": {
        "title": "Physical Activity & Glucose Safety",
        "guidance": (
            "Moderate movement such as brisk walking—especially after meals—and resistance training "
            "can improve insulin sensitivity. Check blood sugar before and after exercise if you use "
            "insulin or insulin secretagogues, and keep fast-acting carbs handy."
        ),
        "related_concepts": ["Physical Activity", "Insulin", "Hypoglycemia"]
    },
    "hypoglycemia_management": {
        "title": "Recognizing & Managing Low Blood Sugar",
        "guidance": (
            "Shakiness, sweating, lightheadedness, hunger, or irritability can signal low blood sugar. "
            "For mild alert cases, many care plans use the 15-15 rule: consume ~15 grams of fast-acting "
            "carbohydrate, wait 15 minutes, and recheck—followed by a balanced snack if still low."
        ),
        "related_concepts": ["Hypoglycemia", "Blood Glucose", "Insulin"]
    },
    "emergency_escalation": {
        "title": "Urgent / Emergency Escalation",
        "guidance": (
            "Symptoms such as confusion, fainting, inability to stay awake, trouble breathing, chest pain, "
            "persistent vomiting, or suspected severe hypoglycemia / diabetic ketoacidosis require "
            "immediate emergency medical care. Call your local emergency number (e.g., 112/108 in India "
            "or 911 in the US) or go to the nearest emergency department right away."
        ),
        "related_concepts": ["Hypoglycemia", "Hyperglycemia / DKA"]
    }
}

In [8]:
EMERGENCY_KEYWORDS = {
    "faint", "fainting", "unconscious", "unresponsive", "seizure", "confusion",
    "confused", "chest pain", "trouble breathing", "breathing", "vomiting",
    "vomit", "severe", "emergency", "collapse", "coma", "fruity breath"
}

UNSUPPORTED_KEYWORDS = {
    "bitcoin", "crypto", "movie", "football", "cricket", "recipe for cake",
    "weather", "stock market", "python code", "wifi", "car repair", "tax"
}

def get_related_facts(intent_name: str):
    rel_concepts = set(knowledge_base.get(intent_name, {}).get("related_concepts", []))
    facts = []
    for src, rel, dst in concept_relationships:
        if src in rel_concepts or dst in rel_concepts:
            facts.append(f"{src} -> [{rel}] -> {dst}")
    return facts[:3]

def run_rules_and_respond(user_query: str, confidence_threshold: float = 0.24):
    q_lower = user_query.strip().lower()
    nlp_out = preprocess_text(user_query)
    tokens_set = set(nlp_out["tokens"]) | set(nlp_out["lemmas"])

    # Rule 1: Emergency escalation rule (highest priority)
    if (tokens_set & EMERGENCY_KEYWORDS) or any(phrase in q_lower for phrase in ["pass out", "passed out", "can't wake", "cant wake", "chest pain"]):
        kb = knowledge_base["emergency_escalation"]
        return {
            "rule_triggered": "RULE_1_EMERGENCY_ESCALATION",
            "intent": "emergency_escalation",
            "confidence": 1.0,
            "response": f"[EMERGENCY ALERT - {kb['title']}]\n{kb['guidance']}",
            "reasoning": get_related_facts("emergency_escalation")
        }

    # Rule 2: Empty / too-short input rule
    if len(nlp_out["lemmas"]) == 0:
        return {
            "rule_triggered": "RULE_2_EMPTY_OR_NOISE_QUERY",
            "intent": "clarification_needed",
            "confidence": 0.0,
            "response": "Please enter a clear diabetes-related question (for example, about blood sugar checks, diet, exercise, or medications).",
            "reasoning": []
        }

    # Rule 3: Unsupported domain query rule
    if (tokens_set & UNSUPPORTED_KEYWORDS) or any(phrase in q_lower for phrase in ["stock market", "movie review", "car engine"]):
        return {
            "rule_triggered": "RULE_3_UNSUPPORTED_QUERY",
            "intent": "unsupported",
            "confidence": 0.0,
            "response": (
                "This assistant focuses on Diabetes Care information (monitoring, nutrition, medications, "
                "physical activity, and hypoglycemia/safety). I can't answer questions outside that scope."
            ),
            "reasoning": []
        }

    # Predict intent via TF-IDF + Classifier
    vec_q = full_vectorizer.transform([nlp_out["processed_str"]])
    probs = final_classifier.predict_proba(vec_q)[0]
    best_idx = int(np.argmax(probs))
    pred_intent = final_classifier.classes_[best_idx]
    confidence = float(probs[best_idx])

    # Rule 4: Low-confidence handling rule
    if confidence < confidence_threshold:
        top_2_idx = np.argsort(probs)[::-1][:2]
        suggestions = ", ".join(final_classifier.classes_[i] for i in top_2_idx)
        return {
            "rule_triggered": "RULE_4_LOW_CONFIDENCE_HANDLING",
            "intent": "low_confidence",
            "confidence": confidence,
            "response": (
                f"I'm not confident enough in how to interpret that query (confidence={confidence:.2f}). "
                f"Could you clarify whether you mean something related to: {suggestions}?"
            ),
            "reasoning": []
        }

    # Rule 5: High/moderate-confidence knowledge retrieval & reasoning rule
    kb = knowledge_base[pred_intent]
    reasoning_links = get_related_facts(pred_intent)
    reasoning_text = "\n".join(f"  - {r}" for r in reasoning_links) if reasoning_links else "  - Direct KB match"
    return {
        "rule_triggered": "RULE_5_KNOWLEDGE_RETRIEVAL_AND_REASONING",
        "intent": pred_intent,
        "confidence": confidence,
        "response": f"[{kb['title']} | Intent: {pred_intent} | Conf: {confidence:.2f}]\n{kb['guidance']}\nDomain Reasoning:\n{reasoning_text}",
        "reasoning": reasoning_links
    }

In [9]:
class DiabetesCognitiveAssistant:
    def __init__(self):
        self.history = []

    def ask(self, query: str):
        result = run_rules_and_respond(query)
        self.history.append({"query": query, **result})
        return result

assistant = DiabetesCognitiveAssistant()

In [10]:
test_queries = [
    # 1. One query per defined intent
    ("1a. Intent [blood_sugar_monitoring]", "What is a normal fasting blood glucose range?"),
    ("1b. Intent [diet_and_nutrition]", "How does carbohydrate counting help manage meals?"),
    ("1c. Intent [medication_and_insulin]", "How should insulin pens be stored properly?"),
    ("1d. Intent [physical_activity]", "Is walking after dinner helpful for glucose control?"),
    ("1e. Intent [hypoglycemia_management]", "What should I do if I feel shaky from low blood sugar?"),
    ("1f. Intent [emergency_escalation]", "Severe confusion and vomiting with high blood sugar"),
    # 2. Differently worded query
    ("2. Differently worded query", "Checking fingerstick glucose two hours after eating"),
    # 3. Unsupported healthcare query
    ("3. Unsupported query", "What is the best strategy for trading stocks or bitcoin?"),
    # 4. Low-confidence query
    ("4. Low-confidence query", "Tell me about seasonal patterns"),
    # 5. Emergency-related query
    ("5. Emergency query", "Someone fainted and is unresponsive after an insulin shot")
]

print("\n==================== TEST SUITE EXECUTION ====================")
for label, q in test_queries:
    out = assistant.ask(q)
    print(f"\n>>> {label}: \"{q}\"")
    print(f"    Rule: {out['rule_triggered']} | Intent: {out['intent']} | Confidence: {out['confidence']:.2f}")
    print(f"    {out['response']}")


==================== TEST SUITE EXECUTION ====================

>>> 1a. Intent [blood_sugar_monitoring]: "What is a normal fasting blood glucose range?"
    Rule: RULE_5_KNOWLEDGE_RETRIEVAL_AND_REASONING | Intent: blood_sugar_monitoring | Confidence: 0.35
    [Blood Glucose Monitoring | Intent: blood_sugar_monitoring | Conf: 0.35]
Regular glucose checks—such as fasting, pre-meal, and 1–2 hours post-meal—help show how food, activity, and medication affect your levels. Keeping a consistent log with timestamps and context makes patterns easier for you and your clinician to review.
Domain Reasoning:
  - Carbohydrates -> [INCREASES] -> Blood Glucose
  - Insulin -> [LOWERS] -> Blood Glucose
  - Physical Activity -> [CAN_REDUCE] -> Blood Glucose

>>> 1b. Intent [diet_and_nutrition]: "How does carbohydrate counting help manage meals?"
    Rule: RULE_5_KNOWLEDGE_RETRIEVAL_AND_REASONING | Intent: diet_and_nutrition | Confidence: 0.25
    [Diabetes Nutrition & Meal Balance | Intent: diet_and_nut

In [11]:
query_input = widgets.Textarea(
    value="How does fiber affect blood sugar?\nI feel dizzy and sweaty\nWhat is the weather today?",
    placeholder="Enter one query per line...",
    description="Queries:",
    layout=widgets.Layout(width="80%", height="90px")
)
run_button = widgets.Button(description="Run Assistant", button_style="primary")
output_box = widgets.Output()

def on_run_clicked(b):
    with output_box:
        clear_output()
        lines = [line.strip() for line in query_input.value.splitlines() if line.strip()]
        for i, q in enumerate(lines, 1):
            res = assistant.ask(q)
            print(f"--- Query {i}: {q} ---")
            print(f"Rule: {res['rule_triggered']} (Confidence: {res['confidence']:.2f})")
            print(res["response"])
            print()

run_button.on_click(on_run_clicked)
display(widgets.VBox([query_input, run_button, output_box]))